# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Auditoría fina y transversal

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

La auditoría parte de tipologías generales de contenido dañino [1] y de abuso dirigido o generalizado [2], pero separa fenómenos implícitos [3], ironía [4] y dependencia de contexto [5]. Para el Perú, `RACISMO_DISCRIMINACION` considera racialización lingüística y política documentada localmente [6]; `ATAQUE_POR_GENERO_IDENTIDAD` se apoya en estudios e informes peruanos sobre violencia de género en línea [7], afectaciones a mujeres y personas LGBTI [8] y léxico lesbofóbico [9]. La frontera de contenido sexual explícito también responde a política de plataforma [10]. Las fusiones, nombres y flags exactos siguen siendo locales.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


## Restauración reproducible del dataset

In [2]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


status,verified_existing
input_key,dataset_5_salidas
path,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/model_ready/v2/dataset_5_salidas.jsonl
sha256,24d3d81d23c00cb1ba27fcca8c2759a97be5c9932ceaeb60be0a4e9a1cfca783
bytes,175177452
archive,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/resultados/colab_bundle/dataset_5_salidas.jsonl.gz


## Configuración y ejecución

In [ ]:
from moderacion_peru.datasets import audit_auxiliary_candidate_metrics,audit_training_snapshot
DATA=ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
OUTPUT=ROOT/'resultados/auditorias/auditoria_finas_flags_v2.json'
PREDICTIVE=ROOT/'resultados/auditorias/calidad_predictiva_auxiliar_observada.json'
audit_result=run_with_progress('Auditoría del snapshot',audit_training_snapshot,DATA,OUTPUT,progress_unit='etapa')
show_result('Auditoría de máscaras, cobertura y consistencia',audit_result,tone='success')
predictive_result=run_with_progress('Auditoría de candidatos',audit_auxiliary_candidate_metrics,[ROOT/'modelos/v2'],PREDICTIVE,progress_unit='etapa')
show_result('Calidad auxiliar disponible',predictive_result,tone='success')

## Referencias

[1] M. Banko, B. MacKeen, and L. Ray, "A Unified Taxonomy of Harmful Content," in Proc. 4th Workshop Online Abuse and Harms, 2020, pp. 125–137, doi: 10.18653/v1/2020.alw-1.16.

[2] Z. Waseem, T. Davidson, D. Warmsley, et al., "Understanding Abuse: A Typology of Abusive Language Detection Subtasks," in Proc. 1st Workshop Abusive Language Online, 2017, pp. 78–84, doi: 10.18653/v1/W17-3012.

[3] M. ElSherief, C. Ziems, D. Muchlinski, et al., "Latent Hatred: A Benchmark for Understanding Implicit Hate Speech," in Proc. EMNLP, 2021, pp. 345–363, doi: 10.18653/v1/2021.emnlp-main.29.

[4] S. Ilić, E. Marrese-Taylor, J. Balazs, et al., "Deep Contextualized Word Representations for Detecting Sarcasm and Irony," in Proc. WASSA, 2018, pp. 2–7, doi: 10.18653/v1/W18-6202.

[5] T. Bourgeade, Z. Li, F. Benamara, et al., "Humans Need Context, What about Machines? Investigating Conversational Context in Abusive Language Detection," in Proc. LREC-COLING, 2024, pp. 8438–8452. [Online]. Available: https://aclanthology.org/2024.lrec-main.740/

[6] V. Zavala and C. Almeida, "‘Motoso y terruco’: ideologías lingüísticas y racialización en la política peruana," Lexis, vol. 46, no. 2, pp. 481–521, 2022, doi: 10.18800/lexis.202202.002.

[7] D. Albornoz and M. Flores, "Conocer para resistir: violencia de género en línea en Perú," Hiperderecho, Lima, Perú, 2018. [Online]. Available: https://hiperderecho.org/tecnoresistencias/wp-content/uploads/2019/01/violencia_genero_linea_peru_2018.pdf

[8] Defensoría del Pueblo del Perú, "Violencia de género contra las mujeres en línea," Documento de Trabajo no. 001-2021-DP/ADM, Aug. 2021. [Online]. Available: https://www.defensoria.gob.pe/wp-content/uploads/2021/08/Documento-de-trabajo-01-Violencia-de-g%C3%A9nero-contra-las-mujeres-en-l%C3%ADnea.pdf

[9] C. M. Lovón-Cueva and M. Lovón-Cueva, "Lesbian Lexicon: The Construction of a Repertoire of Hate in Peruvian Cyberforums," Whatever, vol. 5, no. 1, pp. 43–70, 2022, doi: 10.13131/2611-657X.whatever.v5i1.156.

[10] YouTube, "Política sobre desnudos y contenido sexual," Ayuda de YouTube, 2026. [Online]. Available: https://support.google.com/youtube/answer/2802002?hl=es-419. Accessed: Aug. 5, 2026.